### Middleware 

middleware provides a way to more tightly control what happens inside agent. Middleware is useful for the following:

* Tracking agent behavior
    * Logging
    * Analytics
    * Debugging
* Modifying agent execution
    * Prompt transformation
    * Tool selection control
    * Output formatting
* Improving reliability
    * Retries
    * Fallback mechanisms
    * Early termination logic
* Applying safety and governance
    * Rate limiting
    * Guardrails
    * PII (Personally Identifiable Information) detection and masking

In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

### Summarization Middleware

Summarization Middleware automatically compresses long conversation histories or agent contexts into shorter summaries to reduce token usage while preserving important information.

**Benefits:**
- Reduces context window usage
- Lowers LLM costs
- Improves performance on long conversations
- Retains key information for future interactions

**Use Case:**
When a chat history becomes too large, the middleware summarizes older messages and replaces them with a concise summary before sending the context to the model.

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from dotenv import load_dotenv

load_dotenv()

### messageBased summarization
agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    checkpointer=InMemorySaver(),
    middleware=[SummarizationMiddleware(
        model="groq:llama-3.3-70b-versatile",
        trigger=("messages", 10),
        keep=("messages", 4)
    )],
)

In [3]:
### Run with unique thread(id)

config = {"configurable":{"thread_id":"test-1"}}

In [4]:
### Alternative test data
questions = [
    "what is the capital of France?",
    "what is 10*5?",
    "who is the president of US?",
    "what is the largest mammal?",
    "what is the smallest country in the world?",
    "what is 100/4?",
]

for q in questions:
    response = agent.invoke({"messages": [HumanMessage(content=q)]}, config=config)
    print(f"Question: {q}")
    ###print(f"Answer: {response['messages'][-1].content}")
    print(f"Answer:{response}")
    print(f"Answer:{len(response['messages'])}")

Question: what is the capital of France?
Answer:{'messages': [HumanMessage(content='what is the capital of France?', additional_kwargs={}, response_metadata={}, id='40ef2ec8-7547-411f-9a2e-cbc381ee71d6'), AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 42, 'total_tokens': 50, 'completion_time': 0.011874464, 'completion_tokens_details': None, 'prompt_time': 0.001825129, 'prompt_tokens_details': None, 'queue_time': 0.16171777, 'total_time': 0.013699593}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e8681-9d07-7690-b8e0-2f3a8d7e55dd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 8, 'total_tokens': 50})]}
Answer:2
Question: what is 10*5?
Answer:{'messages': [HumanMessage(content='what is the cap

### Token_Size

In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from dotenv import load_dotenv

load_dotenv()

@tool
def search_hotels(city: str) -> str:
    """Search hotels in a city. Returns hotel names, prices, and ratings."""
    hotels_data = {
        "new york": [
            {"name": "Grand Hotel", "price": "$350/night", "rating": "4.8/5"},
            {"name": "City Inn", "price": "$180/night", "rating": "4.2/5"},
            {"name": "Budget Stay", "price": "$75/night", "rating": "3.5/5"}
        ],
        "paris": [
            {"name": "Hotel Plaza Athenee", "price": "$450/night", "rating": "4.9/5"},
            {"name": "La Reserve Hotel", "price": "$380/night", "rating": "4.7/5"},
            {"name": "The Ritz Paris", "price": "$500/night", "rating": "4.8/5"}
        ],
        "tokyo": [
            {"name": "Park Hyatt Tokyo", "price": "$400/night", "rating": "4.8/5"},
            {"name": "The Ritz-Carlton", "price": "$380/night", "rating": "4.7/5"},
            {"name": "Mandarin Oriental", "price": "$350/night", "rating": "4.6/5"}
        ],
        "sydney": [
            {"name": "Shangri-La Hotel", "price": "$300/night", "rating": "4.6/5"},
            {"name": "The Darling", "price": "$280/night", "rating": "4.5/5"},
            {"name": "Four Seasons", "price": "$320/night", "rating": "4.7/5"}
        ],
        "cairo": [
            {"name": "The Nile Ritz-Carlton", "price": "$250/night", "rating": "4.7/5"},
            {"name": "Four Seasons", "price": "$280/night", "rating": "4.6/5"},
            {"name": "Fairmont Nile City", "price": "$220/night", "rating": "4.5/5"}
        ]
    }
    
    city_lower = city.lower()
    if city_lower in hotels_data:
        hotels = hotels_data[city_lower]
        result = f"Top hotels in {city}:\n"
        for h in hotels:
            result += f"- {h['name']}: {h['price']} | Rating: {h['rating']}\n"
        return result
    return f"No hotels found for {city}."

# Token counter function
def count_tokens(messages):
    total_chars = sum(len(m.content) for m in messages if hasattr(m, 'content') and m.content)
    return total_chars // 4

# Use 70B model with lower threshold for visible summarization
agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[search_hotels],
    system_prompt="You are a hotel search assistant. When the user asks about hotels in a city, you MUST call the search_hotels tool. After receiving tool results, you MUST display the EXACT hotel list from the tool output. NEVER say you cannot answer. NEVER ask follow-up questions. Just show the results as they are.",
    checkpointer=InMemorySaver(),
    middleware=[SummarizationMiddleware(
        model="groq:qwen/qwen3-32b",
        trigger=("tokens", 100),  # Lower threshold to see summarization faster
        keep=("tokens", 50),      # Keep fewer tokens
        token_counter=count_tokens
    )]
)

config = {"configurable": {"thread_id": "test-2"}}

In [6]:
### Run hotel search for multiple cities to test middleware summarization
cities = ["New York", "Paris", "Tokyo", "Sydney", "Cairo"]
prev_msg_count = 0

for city in cities:
    try:
        response = agent.invoke(
            {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
            config=config
        )

        tokens = count_tokens(response["messages"])
        msg_count = len(response["messages"])
        
        print(f"\n{'='*50}")
        print(f"{city} ~{tokens} tokens, {msg_count} messages")
        
        # Detect summarization by checking if first message is a summary
        first_msg = response["messages"][0]
        is_summary = (hasattr(first_msg, 'content') and first_msg.content and 
                     "summary" in first_msg.content.lower())
        
        if is_summary:
            print(f"*** SUMMARIZATION DETECTED! ***")
            print(f"Summary preview: {first_msg.content[:150]}...")
        
        if prev_msg_count > 0 and msg_count < prev_msg_count:
            print(f"Message count dropped: {prev_msg_count} -> {msg_count}")
        
        print(f"{'='*50}")
        
        prev_msg_count = msg_count
        
        # Show all messages
        for i, msg in enumerate(response["messages"]):
            print(f"\n[Message {i}] Type: {msg.type}")
            if hasattr(msg, 'tool_calls') and msg.tool_calls:
                print(f"  Tool call: {msg.tool_calls[0]['name']}({msg.tool_calls[0]['args']})")
            if msg.content:
                content = msg.content[:200] + "..." if len(msg.content) > 200 else msg.content
                print(f"  Content: {content}")
        
    except Exception as e:
        print(f"\n{'='*50}")
        print(f"{city} - Error: {e}")
        print(f"{'='*50}")


New York ~78 tokens, 4 messages

[Message 0] Type: human
  Content: Find hotels in New York

[Message 1] Type: ai
  Tool call: search_hotels({'city': 'New York'})

[Message 2] Type: tool
  Content: Top hotels in New York:
- Grand Hotel: $350/night | Rating: 4.8/5
- City Inn: $180/night | Rating: 4.2/5
- Budget Stay: $75/night | Rating: 3.5/5


[Message 3] Type: ai
  Content: Top hotels in New York:
- Grand Hotel: $350/night | Rating: 4.8/5
- City Inn: $180/night | Rating: 4.2/5
- Budget Stay: $75/night | Rating: 3.5/5

Paris ~904 tokens, 5 messages
*** SUMMARIZATION DETECTED! ***
Summary preview: Here is a summary of the conversation to date:

<think>
Okay, let me try to work through this. The user wants me to extract the most important context...

[Message 0] Type: human
  Content: Here is a summary of the conversation to date:

<think>
Okay, let me try to work through this. The user wants me to extract the most important context from the conversation history provided. The instr...



### Fraction

In [13]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels in a city. Returns hotel names and prices."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[search_hotels],
    system_prompt="You are a hotel assistant. You have ONLY one tool: search_hotels. Use it to answer hotel queries. Do NOT use any other tools.",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 131072  # groq context size
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~68 tokens (0.0519%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='f0901491-edbb-4e71-b25a-5e11a7228338'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for hotels in Paris. Let me check the tools available. There\'s only one function called search_hotels that takes a city parameter. Since the user mentioned Paris, I need to call that function with "Paris" as the city argument. I should make sure the parameters are correctly formatted in JSON. Alright, I\'ll structure the tool call accordingly.\n', 'tool_calls': [{'id': 'zqcqmtv1a', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 101, 'prompt_tokens': 186, 'total_tokens': 287, 'completion_time': 0.168554712, 'completion_tokens_details': {'reasoning_tokens': 76}, 'prompt_time': 0.013419223, 'prompt_tokens_details': None, 'queue_time': 0.

### Human In the Loop MiddleWare
Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

 * High-stakes operations requiring human approval (e.g. database writes, financial transactions).
 * Compliance workflows where human oversight is mandatory.
 * Long-running conversations where human feedback guides the agent.

In [14]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage
from langgraph.types import Command

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[read_email_tool, send_email_tool],
    system_prompt="You are an email assistant. You have two tools: read_email_tool and send_email_tool. Use them to help with email tasks.",
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

config = {"configurable": {"thread_id": "test-approve"}}

# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)
print(f"Paused: {'__interrupt__' in result}")
result

Paused: True


{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='63d442d5-a058-4f9d-8e3f-c57faefb94b0'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the tools available.\n\nThe available functions are read_email_tool and send_email_tool. Since the user is asking to send an email, I should use the send_email_tool. \n\nLooking at the parameters required for send_email_tool: recipient, subject, and body are all required. The user provided all three: recipient is john@test.com, subject is 'Hello', and body is 'How are you?'. \n\nI need to make sure the parameters are correctly formatted as JSON. Let me structure the arguments accordingly. No missing parameters here. \n\nSo, the correct function call would be to send_email_tool with those arguments. I'll generat

In [15]:
# Step 2: Approve
if "__interrupt__" in result:
    print("Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"Result: {result['messages'][-1].content}")

Paused! Approving...
Result: The email has been successfully sent to john@test.com with the subject "Hello". Let me know if you need anything else!


### Reject Example

In [16]:
agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[read_email_tool, send_email_tool],
    system_prompt="You are an email assistant. You have two tools: read_email_tool and send_email_tool. Use them to help with email tasks.",
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

config = {"configurable": {"thread_id": "test-reject"}}

# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

# Step 2: Reject
if "__interrupt__" in result:
    print("Paused! Rejecting...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"Result: {result['messages'][-1].content}")

Paused! Rejecting...
Result: It seems you rejected the email sending. Would you like to edit the email's content, subject, or recipient before sending it again? Or would you prefer to cancel this request? Let me know how you'd like to proceed!


### Edit Example

In [17]:
agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[read_email_tool, send_email_tool],
    system_prompt="You are an email assistant. You have two tools: read_email_tool and send_email_tool. Use them to help with email tasks.",
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)

# Step 2: Edit and approve
if "__interrupt__" in result:
    print("Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",
                            "args": {
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"Result: {result['messages'][-1].content}")

Paused! Editing...
Result: The email has been sent to **correct@email.com** with the subject **"Corrected Subject"**. Let me know if you need further assistance!
